# T5Gemma 2 — test-set evaluation (nominal)

Loads the best checkpoint from `t5gemma2_VAL.ipynb`'s training run and makes
a single forward pass over `data/test.jsonl`. No training happens here.

Writes `test_metrics.json` and `test_predictions.csv` in the same column
layout as `dev_predictions.csv` (including `p_pred` and the per-class
probabilities), so the results notebook can read it identically to the other
three conditions.

In [1]:
# %%
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, glob
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding,
)
from safetensors.torch import load_file

from model import T5Gemma2OrdinalClassifier
from metrics import evaluate_predictions

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)


@dataclass
class CFG:
    model_id: str = "google/t5gemma-2-4b-4b"
    test_path: str = "data/test.jsonl"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 1024
    batch_size: int = 8
    bf16: bool = True
    output_dir: str = "runs/t5gemma2_nominal"   # same dir as the VAL run
    ckpt: str = ""                              # blank -> auto-resolve best_model_checkpoint

cfg = CFG()

W0913 14:51:50.862000 23016 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0913 14:51:50.904000 23016 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
def resolve_ckpt(cfg):
    if cfg.ckpt:
        return os.path.join(cfg.output_dir, cfg.ckpt)
    states = sorted(glob.glob(os.path.join(cfg.output_dir, "checkpoint-*",
                                           "trainer_state.json")))
    if not states:
        raise FileNotFoundError(f"no checkpoints under {cfg.output_dir}")
    best = json.load(open(states[-1])).get("best_model_checkpoint")
    if not best:
        raise ValueError(f"{cfg.output_dir}: no best_model_checkpoint recorded; "
                         "set cfg.ckpt explicitly")
    path = os.path.join(cfg.output_dir, os.path.basename(best))
    if not os.path.isdir(path):
        raise FileNotFoundError(f"best checkpoint {path} was pruned; "
                                "set cfg.ckpt to a surviving one")
    return path

## Data

In [3]:
def read_rows(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


class CEFRDataset(Dataset):
    def __init__(self, path, tokenizer, cfg):
        self.ex, skipped = [], 0
        for row in read_rows(path):
            lab = str(row[cfg.label_col]).strip().upper()
            if lab not in LABEL2ID:
                skipped += 1
                continue
            self.ex.append((str(row[cfg.text_col]), LABEL2ID[lab]))
        print(f"{path}: kept {len(self.ex)}, skipped {skipped} out-of-scope")
        self.tok, self.max_length = tokenizer, cfg.max_length

    def __len__(self):
        return len(self.ex)

    def __getitem__(self, i):
        text, label = self.ex[i]
        enc = self.tok(text, truncation=True, max_length=self.max_length)
        enc["labels"] = label
        return enc


tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
test_ds = CEFRDataset(cfg.test_path, tokenizer, cfg)
collator = DataCollatorWithPadding(tokenizer)

data/test.jsonl: kept 604, skipped 0 out-of-scope


## Load checkpoint and predict

`Trainer` saves a `peft`-wrapped model as an adapter (`adapter_model.safetensors`
+ `adapter_config.json`), not a merged `model.safetensors`/`pytorch_model.bin`.
`PeftModel.from_pretrained` reads `adapter_config.json` to reconstruct the same
`LoraConfig` used at training time and loads the adapter weights onto the base
model; `merge_and_unload()` then bakes the adapter into the base weights so the
rest of this notebook can treat it as a plain model.

In [4]:
class NominalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels.long())
        return (loss, out) if return_outputs else loss


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.asarray(logits).argmax(-1)
    return evaluate_predictions(eval_pred.label_ids, preds)


ckpt = resolve_ckpt(cfg)
print("loading", ckpt)

dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = T5Gemma2OrdinalClassifier(cfg.model_id, num_classes=NUM_CLASSES,
                                  mode="nominal", dtype=dtype)

is_lora_ckpt = os.path.exists(os.path.join(ckpt, "adapter_config.json"))
if is_lora_ckpt:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ckpt)
    model = model.merge_and_unload()
    print("loaded LoRA adapter and merged into base weights")
else:
    sd_path = os.path.join(ckpt, "model.safetensors")
    sd = (load_file(sd_path) if os.path.exists(sd_path)
          else torch.load(os.path.join(ckpt, "pytorch_model.bin"), map_location="cpu"))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"missing {len(missing)} | unexpected {len(unexpected)}")
    if missing:
        print("  e.g.", missing[:5])
    assert not any("head" in k for k in missing), "head weights failed to load"

args = TrainingArguments(
    output_dir="tmp_eval", per_device_eval_batch_size=cfg.batch_size,
    bf16=cfg.bf16, report_to=[], do_train=False,
)
trainer = NominalTrainer(
    model=model, args=args, processing_class=tokenizer,
    data_collator=collator, compute_metrics=compute_metrics,
)

test_out = trainer.predict(test_ds, metric_key_prefix="test")
test_logits = test_out.predictions
if isinstance(test_logits, tuple):
    test_logits = test_logits[0]
test_logits_t = torch.as_tensor(np.asarray(test_logits, dtype=np.float32))
test_preds = test_logits_t.argmax(-1).numpy()
test_gold = test_out.label_ids

probs = torch.softmax(test_logits_t, dim=-1).numpy()
print("test:", json.dumps({k: round(v, 4) for k, v in test_out.metrics.items()
                           if isinstance(v, float)}, indent=2))
print("rows sum to 1:", np.allclose(probs.sum(1), 1.0, atol=1e-4))

loading runs/t5gemma2_nominal\checkpoint-1425


Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

c:\Users\coope\AppData\Local\Programs\Python\Python312\Lib\site-packages\safetensors\torch.py:360: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return f.get_tensors()


loaded LoRA adapter and merged into base weights


test: {
  "test_loss": 0.9567,
  "test_model_preparation_time": 0.0075,
  "test_qwk": 0.8984,
  "test_mae": 0.4619,
  "test_accuracy": 0.5762,
  "test_adjacent_accuracy": 0.9652,
  "test_precision_macro": 0.4578,
  "test_recall_macro": 0.4441,
  "test_f1_macro": 0.442,
  "test_precision_weighted": 0.5581,
  "test_recall_weighted": 0.5762,
  "test_f1_weighted": 0.5606,
  "test_runtime": 9.2259,
  "test_samples_per_second": 65.468,
  "test_steps_per_second": 8.238
}
rows sum to 1: True


## Confusion matrix

In [11]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(test_gold, test_preds, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = (f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
                       if counts[i, j] else "")

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-normalized"),
))
fig.update_xaxes(title_text="Predicted")
fig.update_yaxes(title_text="True", autorange="reversed")
fig.update_layout(
    title=dict(text="T5Gemma 2 nominal — test confusion matrix"),
    font=dict(family="Arial", size=16, color="black"),
    width=620, height=560,
    margin=dict(l=80, r=100, t=100, b=80),
)
fig.show()

## Predictions CSV

Same column layout as `dev_predictions.csv` and the other three conditions,
so the results notebook can read it directly.

In [7]:
srt = np.sort(probs, axis=1)[:, ::-1]
rows = []
for i, (g, p) in enumerate(zip(test_gold, test_preds)):
    g, p = int(g), int(p)
    lo, hi = max(0, p - 1), min(NUM_CLASSES - 1, p + 1)
    rows.append(dict(
        gold=g, pred=p, gold_label=LEVELS[g], pred_label=LEVELS[p],
        correct=int(g == p), adjacent=int(abs(g - p) <= 1),
        p_pred=float(probs[i, p]),
        p_adjacent=float(probs[i, lo:hi + 1].sum()),
        margin=float(srt[i, 0] - srt[i, 1]),
        entropy=float(-(probs[i] * np.log(probs[i] + 1e-12)).sum()),
        exp_level=float((probs[i] * np.arange(NUM_CLASSES)).sum()),
        **{f"p_{lvl}": float(probs[i, j]) for j, lvl in enumerate(LEVELS)},
    ))

test_df = pd.DataFrame(rows)

os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "test_metrics.json"), "w") as f:
    json.dump(test_out.metrics, f, indent=2)
test_df.to_csv(os.path.join(cfg.output_dir, "test_predictions.csv"), index=False)

print(f"Saved to {cfg.output_dir}/test_metrics.json and test_predictions.csv")

Saved to runs/t5gemma2_nominal/test_metrics.json and test_predictions.csv


## Calibration summary

In [8]:
from sklearn.metrics import roc_auc_score


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()
    return float((gap * g.size()).sum() / len(d))


print(f"mean p_pred  {test_df['p_pred'].mean():.3f}")
print(f"mean entropy {test_df['entropy'].mean():.3f}  (max {np.log(NUM_CLASSES):.3f})")
print(f"ECE          {ece(test_df):.4f}")
print()
for col, s in [("p_pred", test_df["p_pred"]), ("margin", test_df["margin"]),
               ("entropy", -test_df["entropy"]), ("p_adjacent", test_df["p_adjacent"])]:
    print(f"{col:12s} exact {roc_auc_score(test_df['correct'], s):.3f}   "
          f"adjacent {roc_auc_score(test_df['adjacent'], s):.3f}")

mean p_pred  0.616
mean entropy 0.883  (max 2.079)
ECE          0.0418

p_pred       exact 0.597   adjacent 0.695
margin       exact 0.594   adjacent 0.659
entropy      exact 0.582   adjacent 0.754
p_adjacent   exact 0.573   adjacent 0.826


In [10]:
edges = np.arange(0.0, 1.01, 0.1)
BANDS = list(zip(edges[:-1], edges[1:],
                 [f"{lo:.1f}\u2013{hi:.1f}" for lo, hi in zip(edges[:-1], edges[1:])]))

n = len(test_df)
band_rows = []
for lo, hi, name in reversed(BANDS):
    s = test_df[(test_df["p_pred"] >= lo) & (test_df["p_pred"] < hi)]
    band_rows.append(dict(
        conf=name, preds=len(s), share=f"{len(s)/n*100:.0f}%",
        acc=round(s["correct"].mean(), 2) if len(s) else None,
        adj=round(s["adjacent"].mean(), 2) if len(s) else None,
    ))

print(pd.DataFrame(band_rows).to_string(index=False))

   conf  preds share  acc  adj
0.9–1.0      2    0% 0.50 1.00
0.8–0.9     53    9% 0.64 1.00
0.7–0.8    103   17% 0.69 1.00
0.6–0.7    169   28% 0.63 0.96
0.5–0.6    155   26% 0.54 0.97
0.4–0.5    105   17% 0.43 0.92
0.3–0.4     17    3% 0.41 0.88
0.2–0.3      0    0%  NaN  NaN
0.1–0.2      0    0%  NaN  NaN
0.0–0.1      0    0%  NaN  NaN
